# WeldSight: SWRD segmentation baseline on a single GPU (Colab T4)

SWRD (Zhao et al., J. Nondestruct. Eval. 2025): ~3,675 seam-weld radiographs, 16-bit, polygon labels for 6 defect classes.
Licence is not stated: research use only, cite the paper, do **not** redistribute the images (keep `data/` out of git).

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
!git clone -b claude/weldsight-stage-1-setup-eeev49 https://github.com/00monaliza/WeldSight.git
%cd WeldSight
!pip -q install uv gdown && uv sync --extra dev

## 1. Download SWRD from the authors' Google Drive folder
Folder linked from https://github.com/bit628/RapidX-Annotator. If `gdown` hits the quota limit, open the folder in the browser, *Add shortcut to My Drive*, mount Drive and copy it to `data/raw/swrd`.

In [ ]:
!mkdir -p data/raw/swrd
!gdown --folder https://drive.google.com/drive/folders/1LNUt101wufTBJpRAgrZAU1h-Tfx629wO -O data/raw/swrd --remaining-ok
# unpack archives if the folder contains zips/rars
!cd data/raw/swrd && for f in $(find . -name '*.zip'); do unzip -q -o "$f" -d "$(dirname $f)"; done
!apt-get -qq install -y unrar > /dev/null; cd data/raw/swrd && for f in $(find . -name '*.rar' ! -name '*part0[2-9]*' ! -name '*part[1-9]*'); do unrar x -o+ -inul "$f" "$(dirname $f)/"; done
!find data/raw/swrd -type f | sed 's/.*\.//' | sort | uniq -c
!find data/raw/swrd -maxdepth 3 -type d | head -30

## 2. Inspect labels before preparing
Every label printed under `WARNING: labels not in class_names` must be mapped in `configs/data/swrd.yaml -> source_opts.label_map`.
If several images belong to one weld (e.g. `xxx_1.tif`, `xxx_2.tif`), set `source_opts.group_regex` so they share a split.

In [ ]:
!uv run weldsight-prepare --config configs/experiment/swrd_baseline.yaml --inspect

In [ ]:
!uv run weldsight-prepare --config configs/experiment/swrd_baseline.yaml
!cat data/processed/swrd_v1/dataset_card.json | head -60
!uv run pytest -q

## 3. Train U-Net (ResNet-34 / ImageNet, Dice + Focal) and evaluate on the real test split

In [ ]:
!uv run weldsight-train --config configs/experiment/swrd_baseline.yaml train.num_workers=2

In [ ]:
import glob
from IPython.display import Image, Markdown, display
run = sorted(glob.glob('runs/swrd_baseline_*'))[-1]
!uv run weldsight-eval --run {run} --split val --uncertainty predictive_entropy
display(Markdown(open(f'{run}/results_test.md').read()))
display(Image(f'{run}/referral_test.png'))

Download the whole `runs/<run>` folder (config, provenance with git commit + data version + seed, history, metrics, figures) and send it back: it is enough to reproduce and report the result.

In [ ]:
!rm -f {run}/best.pt.zip; zip -qr /content/swrd_run.zip {run} -x '*.pt'
from google.colab import files; files.download('/content/swrd_run.zip')